# 3. OCR와 VLM 비교 — OCR 원리 / 한계 / OCR+LLM / OCR+VLM

전통적 OCR 파이프라인(OCR → LLM)과, 이미지를 직접 이해하는 VLM 방식을
같은 영수증 이미지로 비교합니다.

> 이 노트북은 시스템에 **Tesseract-OCR 엔진**이 설치되어 있으면 실제 OCR
> 결과로, 없으면 예시 텍스트로 대체해 진행합니다. (Windows 설치:
> https://github.com/UB-Mannheim/tesseract/wiki)

In [ ]:
from vlm_utils import ask, show

RECEIPT = "data/receipt.png"
show(RECEIPT)

## OCR 원리 (전처리 → 텍스트 검출 → 문자 인식 → 후처리)
`pytesseract`가 있고 Tesseract 엔진이 설치되어 있다면 실제로 돌려봅니다.

In [ ]:
try:
    import pytesseract
    from PIL import Image

    ocr_text = pytesseract.image_to_string(Image.open(RECEIPT), lang="kor+eng").strip()
    print("[실제 OCR 결과]\n", ocr_text)
except Exception as e:
    print(f"Tesseract-OCR을 사용할 수 없어 예시 텍스트로 대체합니다. ({e})")
    ocr_text = (
        "행복마트\n서울시 강남구 테헤란로 123\n2026-07-20 14:32\n"
        "생수 500ml x2 2,000\n삼각김밥 x1 1,500\n초코우유 x1 1,800\n"
        "합계 5,300원"
    )
    print("[예시 OCR 결과]\n", ocr_text)

## OCR의 한계 재현하기
OCR은 저해상도·특수기호에서 흔히 숫자를 오인식합니다(`0`↔`O`, `5`↔`S` 등).
실제 서비스에서 자주 발생하는 오류를 일부러 흉내내 봅니다.

In [ ]:
corrupted_text = ocr_text.replace("5,300원", "S,3OO원")
print(corrupted_text)

## OCR + LLM
OCR로 뽑은 (오염된) 텍스트만 텍스트 전용 LLM에 전달합니다.
여기서는 로컬에 이미 받아둔 `gemma3:4b`를 텍스트 전용으로 사용합니다
(실무에서는 GPT-4, Claude 등 원하는 LLM으로 대체 가능).

In [ ]:
import ollama

ocr_llm_response = ollama.chat(
    model="gemma3:4b",
    messages=[
        {
            "role": "user",
            "content": f"다음은 영수증에서 OCR로 추출한 텍스트야. 총 금액이 얼마인지 알려줘.\n\n{corrupted_text}",
        }
    ],
)
print("[OCR + LLM 결과]\n", ocr_llm_response["message"]["content"])

## OCR + VLM (이미지를 직접 이해)
OCR 단계 없이 이미지를 그대로 VLM에 전달합니다. OCR 오류가 없으므로
숫자 오인식의 영향을 받지 않습니다.

In [ ]:
vlm_response = ask(RECEIPT, "이 영수증에서 총 금액이 얼마야? 숫자만 정확히 답해.")
print("[OCR + VLM(직접) 결과]\n", vlm_response)

### 비교
| 구분 | OCR + LLM | OCR + VLM(직접) |
|---|---|---|
| 입력 | OCR 결과 텍스트 (오염 가능) | 이미지 원본 |
| 이번 실습 결과 | 오인식된 금액을 기반으로 답변 | 이미지에서 정확한 금액(5,300원)을 직접 확인 |
| 시사점 | OCR 오류가 그대로 전파됨 | 레이아웃까지 함께 보고 판단 |